In [ ]:
"""
Minimal Proof-of-Cognition (PoC) Blockchain
------------------------------------------
• Wallet creation / balances
• Token transfer with flat gas fee (50 % burn, 50 % to Network)
• PoC block mining: contributor must answer a math puzzle
• REST API with Flask
"""

from flask import Flask, request, jsonify
import hashlib, random, time, uuid

app = Flask(__name__)

# ─────────────────────────────
# ⛓️  Block and Ledger objects
# ─────────────────────────────
class Block:
    def __init__(self, index, prev_hash, timestamp, data,
                 contributor, transactions, gas_fee,
                 puzzle, solution):
        self.index         = index
        self.prev_hash     = prev_hash
        self.timestamp     = timestamp
        self.data          = data              # human note
        self.contributor   = contributor
        self.transactions  = transactions      # list[dict]
        self.gas_fee       = gas_fee
        self.puzzle        = puzzle
        self.solution      = solution
        self.hash          = self.calc_hash()

    def calc_hash(self):
        record = (
            f"{self.index}{self.prev_hash}{self.timestamp}"
            f"{self.data}{self.contributor}{self.transactions}"
            f"{self.gas_fee}{self.puzzle}{self.solution}"
        )
        return hashlib.sha256(record.encode()).hexdigest()


class Ledger:
    def __init__(self):
        self.base_gas      = 0.01
        self.burn_ratio    = 0.5           # 50 % burned
        self.total_supply  = 1_000_000_000
        self.wallets       = {"Network": 0}
        self.chain         = [self._genesis()]
        self.pending       = {}            # cid ➞ puzzle dict

    # Genesis block
    def _genesis(self):
        return Block(0, "0", time.time(), "Genesis",
                     "Network", [], 0, None, None)

    # Wallet helpers
    def create_wallet(self, wid):
        if wid in self.wallets:
            return False, "Wallet already exists"
        self.wallets[wid] = 1_000
        return True, "Wallet created"

    def balance_of(self, wid):
        if wid not in self.wallets:
            return False, "Wallet not found"
        return True, {"wallet_id": wid, "balance": self.wallets[wid]}

    # Token transfer + gas
    def transfer(self, sender, receiver, amount):
        fee = self.base_gas
        if sender not in self.wallets or receiver not in self.wallets:
            return False, "Invalid wallet ID"
        if self.wallets[sender] < amount + fee:
            return False, "Insufficient balance"

        # update balances
        self.wallets[sender]   -= amount + fee
        self.wallets[receiver] += amount
        burn_amt  = fee * self.burn_ratio
        self.total_supply -= burn_amt
        self.wallets["Network"] += fee - burn_amt

        tx = {
            "from": sender, "to": receiver,
            "amount": amount, "fee": fee,
            "timestamp": time.time()
        }
        return True, tx

    # ─── Proof-of-Cognition ─────────────────
    def issue_challenge(self, contributor):
        a, b = random.randint(10, 99), random.randint(10, 99)
        question = f"{a} + {b} = ?"
        answer   = str(a + b)
        cid = str(uuid.uuid4())
        self.pending[cid] = {
            "question": question,
            "answer":   answer,
            "contributor": contributor
        }
        return cid, question

    def verify_and_mine(self, cid, contributor, answer, txs):
        if cid not in self.pending:
            return False, "Invalid challenge ID"
        chal = self.pending.pop(cid)
        if chal["contributor"] != contributor:
            return False, "Challenge not assigned to contributor"
        if answer.strip() != chal["answer"]:
            return False, "Incorrect answer"

        total_gas = sum(tx["fee"] for tx in txs)
        prev = self.chain[-1]
        block = Block(
            index=len(self.chain),
            prev_hash=prev.hash,
            timestamp=time.time(),
            data="PoC Block",
            contributor=contributor,
            transactions=txs,
            gas_fee=total_gas,
            puzzle=chal["question"],
            solution=answer
        )
        self.chain.append(block)
        return True, block


ledger = Ledger()

# ─────────────────────────────
# 🌐  Flask API
# ─────────────────────────────
@app.post("/wallet/create")
def api_wallet_create():
    wid = request.json.get("wallet_id")
    ok, msg = ledger.create_wallet(wid)
    return (jsonify({"message": msg}), 200) if ok else (jsonify({"error": msg}), 400)

@app.get("/wallet/balance")
def api_wallet_balance():
    wid = request.args.get("wallet_id")
    ok, res = ledger.balance_of(wid)
    return (jsonify(res), 200) if ok else (jsonify({"error": res}), 400)

@app.post("/tx/transfer")
def api_transfer():
    data = request.json
    ok, res = ledger.transfer(data["sender"], data["receiver"], float(data["amount"]))
    return (jsonify({"message": "Transfer OK", "tx": res}), 200) if ok else (jsonify({"error": res}), 400)

@app.post("/poc/challenge")
def api_challenge():
    contributor = request.json.get("contributor")
    cid, question = ledger.issue_challenge(contributor)
    return jsonify({"challenge_id": cid, "question": question})

@app.post("/block/submit")
def api_block_submit():
    data = request.json
    ok, res = ledger.verify_and_mine(
        data["challenge_id"],
        data["contributor"],
        data["answer"],
        data.get("transactions", [])
    )
    return (jsonify({"message": "Block accepted", "block": res.__dict__}), 201) if ok else (jsonify({"error": res}), 400)

@app.get("/chain")
def api_chain():
    return jsonify({"length": len(ledger.chain),
                    "chain": [b.__dict__ for b in ledger.chain]})

# ─────────────────────────────
if __name__ == "__main__":
    ledger.create_wallet("Network")  # reserve
    app.run(port=5000, debug=False)


 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on http://127.0.0.1:5000
INFO:werkzeug:Press CTRL+C to quit
